# Continuous arXiv Monitoring

This notebook automatically fetches and catalogs new condensed matter physics papers from arXiv.

## Features
- Fetch latest papers from arXiv condensed matter categories
- Parse paper metadata (title, authors, abstract, DOI)
- Append new papers to `data/raw/recent_arxiv/README.md`
- Track which papers have been processed

## Usage
Run this notebook daily/weekly to keep your training corpus up-to-date with the latest research.

In [ ]:
import requests
import re
from datetime import datetime
from pathlib import Path
import json

# Configuration
ARXIV_API_BASE = "http://export.arxiv.org/api/query"
CATEGORIES = ["cond-mat.mtrl-sci", "cond-mat.str-el", "cond-mat.quant-gas", "cond-mat.mes-hall"]
MAX_RESULTS = 10  # Fetch top 10 papers per category
TRACKING_FILE = Path("data/raw/recent_arxiv/processed.json")
README_FILE = Path("data/raw/recent_arxiv/README.md")

print(f"🔍 Monitoring arXiv for categories: {', '.join(CATEGORIES)}")

In [ ]:
def fetch_arxiv_papers(category, max_results=10):
    """Fetch latest papers from arXiv for a given category."""
    params = {
        'search_query': f'cat:{category}',
        'start': 0,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending'
    }
    
    response = requests.get(ARXIV_API_BASE, params=params)
    response.raise_for_status()
    
    papers = []
    for entry in response.text.split('<entry>'):
        if '<id>' not in entry:
            continue
        
        paper = {}
        
        # Extract ID (arXiv number)
        id_match = re.search(r'<id>([^<]+)</id>', entry)
        if id_match:
            paper['id'] = id_match.group(1).strip()
        
        # Extract title
        title_match = re.search(r'<title>([^<]+)</title>', entry)
        if title_match:
            paper['title'] = title_match.group(1).strip().replace('\n', ' ')
        
        # Extract abstract
        abstract_match = re.search(r'<summary>([^<]+)', entry)
        if abstract_match:
            paper['abstract'] = abstract_match.group(1).strip()[:2000]  # Truncate long abstracts
        
        # Extract authors
        authors = re.findall(r'<name>([^<]+)</name>', entry)
        if authors:
            paper['authors'] = ', '.join(authors[:3]) + ' et al.' if len(authors) > 3 else ', '.join(authors)
        
        # Extract published date
        published_match = re.search(r'<published>([^<]+)</published>', entry)
        if published_match:
            paper['published'] = published_match.group(1)[:10]  # YYYY-MM-DD
        
        if 'id' in paper and 'title' in paper:
            papers.append(paper)
    
    return papers

print("✅ Function defined: fetch_arxiv_papers()")

In [ ]:
def load_processed_papers():
    """Load list of already processed paper IDs."""
    if TRACKING_FILE.exists():
        with open(TRACKING_FILE, 'r') as f:
            return json.load(f)
    return []

def save_processed_papers(papers):
    """Save list of processed paper IDs."""
    TRACKING_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(TRACKING_FILE, 'w') as f:
        json.dump(papers, f, indent=2)

print("✅ Functions defined: load_processed_papers(), save_processed_papers()")

In [ ]:
def format_paper_entry(paper, index):
    """Format a paper as markdown for README."""
    arxiv_id = paper['id'].split('/')[-1]
    
    return f"""## Paper {index}: {paper['title']}

**Authors:** {paper.get('authors', 'Unknown')}

**arXiv:** [{arxiv_id}](https://arxiv.org/abs/{arxiv_id})

**Date:** {paper.get('published', 'Unknown')}

**Abstract:** {paper.get('abstract', 'No abstract available')}

**Full Text:** [View PDF](https://arxiv.org/pdf/{arxiv_id})

---
"""

print("✅ Function defined: format_paper_entry()")

In [ ]:
# Step 1: Load already processed papers
processed_ids = load_processed_papers()
print(f"📊 Already processed: {len(processed_ids)} papers")

# Step 2: Fetch new papers
all_new_papers = []
for category in CATEGORIES:
    print(f"\n📚 Fetching from {category}...")
    papers = fetch_arxiv_papers(category, MAX_RESULTS)
    
    # Filter out already processed papers
    new_papers = [p for p in papers if p['id'] not in processed_ids]
    all_new_papers.extend(new_papers)
    
    print(f"  Found {len(papers)} papers, {len(new_papers)} new")

print(f"\n✨ Total new papers to add: {len(all_new_papers)}")

In [ ]:
if len(all_new_papers) > 0:
    # Step 3: Append new papers to README
    print("\n📝 Adding new papers to README.md...")
    
    # Get current paper count
    current_count = len(processed_ids)
    
    # Format new papers
    new_entries = []
    for i, paper in enumerate(all_new_papers, start=1):
        index = current_count + i
        entry = format_paper_entry(paper, index)
        new_entries.append(entry)
    
    # Read current README
    if README_FILE.exists():
        content = README_FILE.read_text(encoding='utf-8')
        # Remove "Last updated" footer
        content = re.sub(r'\n\n\*Last updated:.*', '', content)
    else:
        content = "# Recent arXiv Papers - Condensed Matter Physics\n\n"
    
    # Append new papers
    content += "\n".join(new_entries)
    content += f"\n\n*Last updated: {datetime.now().strftime('%Y-%m-%-%d')}*\n"
    content += "*These papers represent cutting-edge research in condensed matter physics.*\n"
    
    # Write updated README
    README_FILE.write_text(content, encoding='utf-8')
    print(f"✅ Added {len(all_new_papers)} papers to README.md")
    
    # Update processed list
    new_ids = [p['id'] for p in all_new_papers]
    processed_ids.extend(new_ids)
    save_processed_papers(processed_ids)
    print(f"✅ Updated tracking file ({len(processed_ids)} total papers)")
else:
    print("\n✅ No new papers to add!")

In [ ]:
# Summary
print("\n" + "="*50)
print("📊 MONITORING SUMMARY")
print("="*50)
print(f"Categories monitored: {', '.join(CATEGORIES)}")
print(f"Papers fetched: {len(all_new_papers)}")
print(f"Total papers tracked: {len(processed_ids)}")
print(f"Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("="*50)

if len(all_new_papers) > 0:
    print("\n🎯 Next steps:")
    print("1. Review the added papers in data/raw/recent_arxiv/README.md")
    print("2. Commit changes: git add data/raw/recent_arxiv/README.md")
    print("3. Commit: git commit -m 'Add [date] arXiv papers'")
    print("4. Push: git push origin main")